# Task 2 / ICL（事例設計部門）ベースライン

LLM とプロンプトは**運営が固定**します（モデル: gpt-5.4-nano・temperature=0・seed 固定）。参加者が動かせるのは、**各評価サンプルに与える few-shot 事例（最大5件）の設計だけ**です。事例は train からの選択に加え、**加工・自作も可能**です。

この notebook では、方向性の異なる2つのベースラインを実装します。

| ベースライン | 方向性 | 学べること |
|---|---|---|
| A: 例示なし（zero-shot）＋固定パイプラインの理解 | まず土台を知る | OpenAI API の使い方・固定プロンプトの中身 |
| B: 事例選択（静的ランダム / kNN 適応選択） | どの事例を見せるかを選ぶ | 埋め込み検索・提出のスクリプト化 |

**データの使い方（この notebook の流れ）**

```
train.jsonl (1,000問・ラベル付き)      → 事例の供給源
dev_labeled.jsonl (200問・ラベル付き)  → 自前検証・戦略のチューニング（API キーがあれば）
dev_leaderboard.jsonl (100問・ラベルなし) → 最後に各サンプルへの事例割当を出力して提出
```

**提出形式**（1行1評価サンプルの JSONL。事例は内容を直接記載）:
```json
{"id": "dev-003", "exemplars": [
  {"citation_context": "... [CITE] ...", "cited_paper_id": "Vol21No05_02", "label": 1}
]}
```

**レギュレーションの要点**
- 事例は1評価サンプルあたり**5件以下**（0件も可）
- `citation_context` は **500文字以内**・**`[CITE]` をちょうど1個**含む
- `cited_paper_id` は配布 `papers.jsonl` に存在する ID のみ、`label` は整数の 0/1
- 公式スコアはリーダーボードと最終評価のみ（自前 API での事前検証は自由）

**実行環境**
- **Google Colab では GPU ランタイムを推奨**（メニュー［ランタイム］→［ランタイムのタイプを変更］→ **T4 GPU**）。
  ベースラインBの埋め込み計算が CPU では数十分かかります（GPU なら数分）
- データは「環境設定」の次のセルで自動取得します（Colab では配布リポジトリをクローン、
  ローカルでは `data/`・`../data` を自動探索）

In [ ]:
# 必要なライブラリ（初回のみ）
%pip install -q openai sentence-transformers numpy

In [ ]:
# データの取得（Google Colab 用）: 配布リポジトリをクローンする。
# ローカルで配布リポジトリの中から実行している場合、このセルは何もしません。
![ -d data ] || [ -d ../data ] || git clone -q https://github.com/YANS-official/yans-2026-hackathon

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np

# 配布データの場所を自動で探す（./data → ../data の順。環境変数 YANS_DATA_DIR でも指定可能）
# ローカルの Jupyter は notebook のあるフォルダが作業ディレクトリになるため、
# リポジトリ直下の data/ は notebooks/ から見ると ../data になる。
_candidates = ([Path(os.environ["YANS_DATA_DIR"])] if os.environ.get("YANS_DATA_DIR") else []) + [
    Path("data"), Path("../data"), Path("yans-2026-hackathon/data")]
DATA_DIR = next((p for p in _candidates if (p / "train.jsonl").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "配布データ（data/）が見つかりません。notebook と同じ階層か1つ上に data/ を置くか、"
        "環境変数 YANS_DATA_DIR でデータの場所を指定してください")
print(f"データディレクトリ: {DATA_DIR.resolve()}")

# --- 読み込む前に、ファイルの存在とサイズを検査する ---
# Colab へのアップロードが完了する前にセルを実行すると、途中までのファイルを読んで
# UnicodeDecodeError になる。先にサイズを検査して、原因が分かる形で検出する。
MIN_BYTES = {"papers.jsonl": 55_000_000, "train.jsonl": 400_000,
             "dev_labeled.jsonl": 80_000, "dev_leaderboard.jsonl": 40_000}
for name, min_bytes in MIN_BYTES.items():
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(
            f"{path} が見つかりません。data/ の場所（DATA_DIR）を確認してください")
    size = path.stat().st_size
    print(f"{name:24s} {size / 1e6:6.2f} MB")
    if size < min_bytes:
        raise RuntimeError(
            f"{name} が本来のサイズ（目安 {min_bytes / 1e6:.1f}MB 以上）より小さく、"
            f"途中で切れています（現在 {size:,} bytes）。アップロードの転送が完了する前に"
            "実行したか、転送が途中で失敗しています。アップロードをやり直し、"
            "ファイルペインの進行表示が消えてからこのセルを再実行してください")

def read_jsonl(path):
    try:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    except (UnicodeDecodeError, json.JSONDecodeError) as e:
        raise RuntimeError(
            f"{path} の読み込みに失敗しました。ファイルが途中で切れている可能性が高いです。"
            "アップロードをやり直し、転送完了を待ってから再実行してください") from e

train = read_jsonl(DATA_DIR / "train.jsonl")            # 事例の供給源（1,000問・ラベル付き）
dev = read_jsonl(DATA_DIR / "dev_labeled.jsonl")        # 自前検証・チューニング用（200問・ラベル付き）
lb = read_jsonl(DATA_DIR / "dev_leaderboard.jsonl")     # リーダーボード対象（100問・ラベルなし）
papers = {p["paper_id"]: p for p in read_jsonl(DATA_DIR / "papers.jsonl")}
print(f"\ntrain={len(train)}  dev_labeled={len(dev)}  dev_leaderboard={len(lb)}")

## 1. 固定プロンプトを理解する

運営サーバでは、以下のテンプレート（事前共有されている `prompt_template.py` と同一）に皆さんの事例を挿入して gpt-5.4-nano に判定させます。**何がどう挿入されるかを知ることが、事例設計の出発点**です。

In [ ]:
SYSTEM_PROMPT = """あなたは学術論文の引用の妥当性を判定する専門家です。
与えられた「引用文脈」中の [CITE] の位置での引用が、示された「引用先論文」の内容として
妥当かどうかを判定してください。

判定基準:
- 妥当(1): 引用文脈が述べている内容と、引用先論文の内容が実際に合致している
- 不適切(0): 内容の不一致、トピックのずれ、粒度の不一致、存在しない主張の帰属、
  時期の不適切さ、のいずれかに該当する

few-shot の例が与えられる場合は、それらの判定パターンを参考にしてください。
出力は 0 または 1 の整数のみとし、他の文字は含めないでください。"""

def _paper_block(paper):
    if paper is None:
        return "(論文情報が見つかりません)"
    return (f"タイトル: {paper.get('title', '')}\n著者: {paper.get('author', '')}\n"
            f"年: {paper.get('year', '')}\n概要: {paper.get('abstract', '')}")

def build_exemplar_block(ex):
    return (f"引用文脈: {ex.get('citation_context', '')}\n引用先論文:\n"
            f"{_paper_block(papers.get(ex.get('cited_paper_id')))}\n判定: {ex.get('label')}")

def build_question_block(q):
    return (f"引用文脈: {q.get('citation_context', '')}\n引用先論文:\n"
            f"{_paper_block(papers.get(q.get('cited_paper_id')))}\n判定: ")

def build_messages(question, exemplars):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if exemplars:
        examples_text = "\n\n".join(build_exemplar_block(e) for e in exemplars)
        messages.append({"role": "user", "content": f"以下は判定の例です:\n\n{examples_text}"})
        messages.append({"role": "assistant", "content": "了解しました。例を参考に判定します。"})
    messages.append({"role": "user", "content": build_question_block(question)})
    return messages

# 例示なし（zero-shot）のプロンプトを1問分見てみる
for m in build_messages(dev[0], []):
    print(f"--- {m['role']} ---")
    print(m["content"][:300], "..." if len(m["content"]) > 300 else "")

## 2. ベースラインA: 例示なし（zero-shot）と自前検証の道具

最初のベースラインは「例示なし」（`exemplars` を空にする）です。固定モデルの素の実力＝**超えるべき基準線**になります。

あわせて、自分の API キーで運営と同じ処理を手元再現し、`dev_labeled` で答え合わせする関数を用意します。**キーがなくても提出はできる**のでスキップ可能です。

> 費用の目安: gpt-5.4-nano で数十問なら数円程度です。キーは `OPENAI_API_KEY` 環境変数に設定してください（Colab では左のシークレット機能で設定できます）。

In [ ]:
client = None
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("APIキーを確認しました")
else:
    print("OPENAI_API_KEY が未設定のため、API を使うセルはスキップされます（提出ファイルは作れます）")

def judge(question, exemplars, model="gpt-5.4-nano"):
    """固定パイプラインと同じ呼び出しで 0/1 を返す。"""
    resp = client.chat.completions.create(
        model=model, messages=build_messages(question, exemplars),
        temperature=0.0, seed=42, max_completion_tokens=4096)
    text = (resp.choices[0].message.content or "").strip()
    for ch in text:
        if ch in "01":
            return int(ch)
    return 1  # パース不能時のフォールバック

def local_eval(assignments, records, n=20):
    """dev_labeled のうち n 問でざっくり Accuracy を測る（自前検証用）。"""
    sub = records[:n]
    correct = sum(judge(r, assignments.get(r["id"], [])) == int(r["label"]) for r in sub)
    return correct / len(sub)

if client:
    acc = local_eval({}, dev, n=5)
    print(f"zero-shot の dev_labeled 先頭5問 Accuracy: {acc:.2f}  ※ごく少数の動作確認")

## 3. ベースラインB: 事例選択

### B-1: 静的選択 — 全問に同じ5件（まずは無作為から）

train からラベル均衡（正例2＋負例3）で無作為に5件選び、全問共通で与えます。**「どの5件か」で結果は変わります**。シードを変えて dev_labeled で比べてみましょう。

In [ ]:
def pick_random_exemplars(n=5, seed=42):
    rng = random.Random(seed)
    pos = [r for r in train if int(r["label"]) == 1]
    neg = [r for r in train if int(r["label"]) == 0]
    chosen = rng.sample(pos, n // 2) + rng.sample(neg, n - n // 2)
    rng.shuffle(chosen)
    return [{"citation_context": r["citation_context"],
             "cited_paper_id": r["cited_paper_id"], "label": int(r["label"])} for r in chosen]

static5 = pick_random_exemplars(seed=42)
print("選ばれた事例のラベル:", [e["label"] for e in static5])

### B-2: 適応的選択 — 埋め込み kNN でクエリごとに似た事例を選ぶ

評価サンプルごとに、**引用文脈が似ている train 事例**を埋め込みで検索して与えます。ここからは事例割当をスクリプトで自動生成します（最終評価は test 公開から提出締切まで30分のため、自動化を前提にしてください）。

> e5 系モデルは検索クエリに `query: `、検索対象に `passage: ` の prefix を付けて使うのが正式な使い方です。また、選んだ事例の**ラベル構成**（正例と負例のバランス）も判定に影響します。`balanced` を切り替えて試してみてください。

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sbintuitions/modernbert-ja-310m"  # お試しは "intfloat/multilingual-e5-small" でも可
device = "cuda" if torch.cuda.is_available() else None  # None = 自動選択
if device is None and not (getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()):
    print("警告: GPU が見つかりません。CPU では埋め込み計算に時間がかかります。")
    print("Colab では［ランタイム］→［ランタイムのタイプを変更］→ T4 GPU を推奨します。")

model = SentenceTransformer(MODEL_NAME, device=device)  # 初回はダウンロードに数分
print("計算デバイス:", model.device)
train_emb = model.encode([f"passage: {r['citation_context']}" for r in train],
                         normalize_embeddings=True, show_progress_bar=True)

def knn_exemplars(query_vec, n=5, balanced=True):
    order = (train_emb @ query_vec).argsort()[::-1]
    if balanced:  # 正例2+負例3 をラベル別の類似上位から
        pos = [j for j in order if int(train[j]["label"]) == 1][: n // 2]
        neg = [j for j in order if int(train[j]["label"]) == 0][: n - n // 2]
        rank = {j: k for k, j in enumerate(order)}
        idx = sorted(pos + neg, key=lambda j: rank[j])
    else:
        idx = list(order[:n])
    return [{"citation_context": train[j]["citation_context"],
             "cited_paper_id": train[j]["cited_paper_id"], "label": int(train[j]["label"])} for j in idx]

## 4. dev_labeled で戦略をチューニングする（API キーがある場合）

どの戦略が良いかは、**dev_labeled で自前検証**して選べます。ここでは3戦略（zero / static / knn）を少数サンプルで比較します。

> 注意: 少数サンプルの比較には API 実行の揺らぎも乗ります。あくまで目安とし、有望な戦略は問数を増やして確かめましょう。公式スコアはリーダーボードのみです。

In [ ]:
if client:
    N = 15
    dev_sub = dev[:N]
    dev_emb = model.encode([f"query: {r['citation_context']}" for r in dev_sub],
                           normalize_embeddings=True, show_progress_bar=False)
    strategies = {
        "zero(例示なし)   ": {},
        "static(無作為5件)": {r["id"]: static5 for r in dev_sub},
        "knn(適応選択)    ": {r["id"]: knn_exemplars(v) for r, v in zip(dev_sub, dev_emb)},
    }
    for name, asg in strategies.items():
        print(f"{name} dev先頭{N}問 Accuracy: {local_eval(asg, dev, n=N):.2f}")
else:
    print("APIキーがないためスキップ（戦略の比較はリーダーボードでも行えます）")

## 5. 最終出力: dev_leaderboard への事例割当を保存する

最後に、`dev_leaderboard.jsonl` の**各サンプルに対する事例割当**を提出形式（JSONL）で書き出します。3戦略それぞれのファイルを作るので、リーダーボードに提出して比べられます。

あわせて、**レギュレーションの形式チェック**（5件以下・500文字以内・`[CITE]`1個・実在ID・label 0/1）を提出前に必ず通しましょう。運営側でも同じ検査が走ります。

In [ ]:
def save_submission(path, assignments):
    """assignments: {評価サンプルid: 事例リスト} を提出形式で書き出す。"""
    with open(path, "w", encoding="utf-8") as f:
        for rec in lb:
            exs = assignments.get(rec["id"], [])
            f.write(json.dumps({"id": rec["id"], "exemplars": exs}, ensure_ascii=False) + "\n")
    print(f"{path} を書き出しました（{len(lb)}行）")

def validate_submission(path):
    lb_ids = {r["id"] for r in lb}
    n_err = 0
    for i, line in enumerate(open(path, encoding="utf-8"), 1):
        rec = json.loads(line)
        errs = []
        if rec.get("id") not in lb_ids:
            errs.append("id が評価対象にない")
        exs = rec.get("exemplars", [])
        if len(exs) > 5:
            errs.append("事例が5件を超過")
        for j, e in enumerate(exs):
            ctx = e.get("citation_context", "")
            if len(ctx) > 500: errs.append(f"事例{j}: 500文字超過")
            if ctx.count("[CITE]") != 1: errs.append(f"事例{j}: [CITE] が1個でない")
            if e.get("cited_paper_id") not in papers: errs.append(f"事例{j}: 不明な cited_paper_id")
            if e.get("label") not in (0, 1): errs.append(f"事例{j}: label が 0/1 でない")
        if errs:
            n_err += 1
            print(f"  行{i}: {'; '.join(errs)}")
    print(f"{path}: {'OK（違反なし）' if n_err == 0 else f'{n_err}行に違反あり'}")

# dev_leaderboard の各サンプルへの事例割当を生成して保存（この notebook の最終成果物）
lb_emb = model.encode([f"query: {r['citation_context']}" for r in lb],
                      normalize_embeddings=True, show_progress_bar=True)

submissions = {
    "submission_task2_zero.jsonl": {},
    "submission_task2_static.jsonl": {r["id"]: static5 for r in lb},
    "submission_task2_knn.jsonl": {r["id"]: knn_exemplars(v) for r, v in zip(lb, lb_emb)},
}
for path, asg in submissions.items():
    save_submission(path, asg)
    validate_submission(path)

## 6. 改善の方向性

3つの提出（zero / static / knn）をリーダーボードに出すと、**事例の与え方でスコアが動く**ことを確認できます。改善の例:

1. **誤り分析**: `local_eval` の中身を少し変えれば「どの問題を間違えたか」が取れます。dev_labeled の誤答を**自分の目で読み**、どんな誤りが多いかを把握する
2. **事例の加工・自作**: 事例は選ぶだけでなく、書き換え・自作もできます。誤りパターンを狙った事例を規定内（500字・`[CITE]`1個・実在 ID）で自作して混ぜる
3. **構成の設計**: 正例と負例の比率・並び順・何を教える例か、を意図を持って組む
4. **自動化**: 最終評価は 17:30 の test 公開から **18:00 提出締切まで30分**。この notebook の§5をスクリプト化しておけば、当日は `lb` を test に差し替えて実行するだけです

ヒント: 「とりあえず似た事例を入れる」だけが正解とは限りません。**モデルが何を間違え、事例で何を教えられるか**を考えるのが事例設計です。